# EcoShield AI — Streamlit Prototipi

Bu notebook Görev 8 Streamlit uygulamasının ön kontrolünü ve başlatma komutunu sağlar.

Uygulama:

- Görev 6'da seçilen `cat_d8_balanced` modelini kullanır.
- Validation'da sabitlenen threshold değerini değiştirmez.
- Tek işlem JSON girdisi ve toplu CSV tahmini destekler.
- Transaction ve identity CSV dosyalarını `TransactionID` üzerinden left join edebilir.
- Büyük tahminlerde chunk, ilerleme, işlenen satır, süre ve ETA gösterir.
- Model performansını Görev 7 final test çıktılarından gösterir.

Model eğitimi, threshold tuning veya test değerlendirmesi bu görevde tekrar yapılmaz.


## 1. Dosya ve paket kontrolü

In [4]:
import importlib.util
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "streamlit_app.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "streamlit_app.py").exists():
        NOTEBOOK_DIR = candidate

PROJECT_ROOT = NOTEBOOK_DIR.parent
APP_PATH = NOTEBOOK_DIR / "streamlit_app.py"
MODEL_PATH = PROJECT_ROOT / "models" / "heavy" / "optimized_single_heavy_model.joblib"
SELECTION_PATH = PROJECT_ROOT / "outputs" / "metrics" / "selected_single_heavy_model.csv"
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "metadata" / "common_cache_manifest.json"
FINAL_METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics" / "final_test_comparison.csv"

required_packages = ["streamlit", "catboost", "joblib", "numpy", "pandas", "pyarrow"]
missing_packages = [
    package
    for package in required_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing_packages))

required_files = [APP_PATH, MODEL_PATH, SELECTION_PATH, MANIFEST_PATH, FINAL_METRICS_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Eksik Görev 8 dosyaları:\n" + "\n".join(str(path) for path in missing_files)
    )

subprocess.run([sys.executable, "-m", "py_compile", str(APP_PATH)], check=True)
print("Streamlit uygulaması Python sözdizimi: OK")
print("Final model:", MODEL_PATH.relative_to(PROJECT_ROOT))
print("Uygulama:", APP_PATH.relative_to(PROJECT_ROOT))


Streamlit uygulaması Python sözdizimi: OK
Final model: models\heavy\optimized_single_heavy_model.joblib
Uygulama: notebooks\streamlit_app.py


## 2. Başlatma komutu

In [5]:
launch_command = f'"{sys.executable}" -m streamlit run "{APP_PATH}"'
print("VS Code terminalinde çalıştırın:")
print(launch_command)
print("\nStreamlit varsayılan olarak http://localhost:8501 adresini açacaktır.")


VS Code terminalinde çalıştırın:
"c:\Users\pc\anaconda3\envs\torchcuda\python.exe" -m streamlit run "c:\Users\pc\Desktop\YZTA-Bootcamp-2026\notebooks\streamlit_app.py"

Streamlit varsayılan olarak http://localhost:8501 adresini açacaktır.


## 3. Kullanım

Terminal komutunu çalıştırdıktan sonra:

1. **Tek işlem** sekmesinde ortak feature şemasındaki alanları JSON olarak girin.
2. **Toplu CSV** sekmesinde birleştirilmiş CSV veya ayrı transaction/identity CSV dosyalarını yükleyin.
3. Tahminleri başlatın ve sonuç CSV'sini indirin.
4. **Model performansı** sekmesinde final test metriklerini ve görsellerini inceleyin.

`isFraud` kolonu yüklenirse tahmin girdisinden çıkarılır. Uygulama hedef kolonunu model girdisi olarak kullanmaz.


# Görev 8 tamamlanma koşulları

- [x] Final D8 Balanced model dosyası yüklenir.
- [x] Validation'da seçilen threshold sabit kullanılır.
- [x] Tek işlem ve toplu CSV tahmini desteklenir.
- [x] Transaction–identity left join protokolü korunur.
- [x] Feature sırası ortak şemaya göre doğrulanır.
- [x] Kategorik eksikler `__MISSING__`, sayısal eksikler `NaN` olarak hazırlanır.
- [x] Uzun tahminlerde ilerleme, satır sayısı, süre ve ETA gösterilir.
- [x] Tahmin sonuçları CSV olarak indirilebilir.
- [x] Final test metrikleri ve görselleri gösterilir.
- [x] Model eğitimi, threshold tuning ve test değerlendirmesi tekrarlanmaz.


In [6]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

test_path = project_root / "data" / "processed" / "catboost" / "test.parquet"

test_df = pd.read_parquet(test_path)

sample_json = test_df.iloc[24586].where(
    test_df.iloc[24586].notna(),
    None
).to_dict()

import json
print(json.dumps(sample_json, ensure_ascii=False, indent=2))

{
  "TransactionDT": 3455007,
  "TransactionAmt": 280.0,
  "ProductCD": "W",
  "card1": "11162",
  "card2": "346.0",
  "card3": "150.0",
  "card4": "mastercard",
  "card5": "224.0",
  "card6": "debit",
  "addr1": "299.0",
  "addr2": "87.0",
  "dist1": null,
  "dist2": null,
  "P_emaildomain": "gmail.com",
  "R_emaildomain": "__MISSING__",
  "C1": 1.0,
  "C2": 1.0,
  "C3": 0.0,
  "C4": 0.0,
  "C5": 0.0,
  "C6": 2.0,
  "C7": 0.0,
  "C8": 0.0,
  "C9": 1.0,
  "C10": 0.0,
  "C11": 1.0,
  "C12": 0.0,
  "C13": 4.0,
  "C14": 1.0,
  "D1": 68.0,
  "D2": 68.0,
  "D3": 57.0,
  "D4": 57.0,
  "D5": 57.0,
  "D6": null,
  "D7": null,
  "D8": null,
  "D9": null,
  "D10": 68.0,
  "D11": null,
  "D12": null,
  "D13": null,
  "D14": null,
  "D15": 68.0,
  "M1": "__MISSING__",
  "M2": "__MISSING__",
  "M3": "__MISSING__",
  "M4": "M1",
  "M5": "F",
  "M6": "T",
  "M7": "__MISSING__",
  "M8": "__MISSING__",
  "M9": "__MISSING__",
  "V1": null,
  "V2": null,
  "V3": null,
  "V4": null,
  "V5": null,
  "V6": 

In [5]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

test_path = project_root / "data" / "processed" / "catboost" / "test.parquet"

test_df = pd.read_parquet(test_path)

sample_json = test_df.iloc[36585].where(
    test_df.iloc[36585].notna(),
    None
).to_dict()

import json
print(json.dumps(sample_json, ensure_ascii=False, indent=2))

{
  "TransactionDT": 5773828,
  "TransactionAmt": 445.0,
  "ProductCD": "W",
  "card1": "11106",
  "card2": "100.0",
  "card3": "150.0",
  "card4": "visa",
  "card5": "226.0",
  "card6": "credit",
  "addr1": "126.0",
  "addr2": "87.0",
  "dist1": null,
  "dist2": null,
  "P_emaildomain": "gmail.com",
  "R_emaildomain": "__MISSING__",
  "C1": 7.0,
  "C2": 4.0,
  "C3": 0.0,
  "C4": 0.0,
  "C5": 0.0,
  "C6": 4.0,
  "C7": 0.0,
  "C8": 0.0,
  "C9": 1.0,
  "C10": 0.0,
  "C11": 3.0,
  "C12": 0.0,
  "C13": 8.0,
  "C14": 6.0,
  "D1": 2.0,
  "D2": 2.0,
  "D3": 0.0,
  "D4": 305.0,
  "D5": 2.0,
  "D6": null,
  "D7": null,
  "D8": null,
  "D9": null,
  "D10": 508.0,
  "D11": null,
  "D12": null,
  "D13": null,
  "D14": null,
  "D15": 305.0,
  "M1": "__MISSING__",
  "M2": "__MISSING__",
  "M3": "__MISSING__",
  "M4": "M0",
  "M5": "T",
  "M6": "T",
  "M7": "__MISSING__",
  "M8": "__MISSING__",
  "M9": "__MISSING__",
  "V1": null,
  "V2": null,
  "V3": null,
  "V4": null,
  "V5": null,
  "V6": null,


In [ ]:
import pandas as pd
from pathlib import Path

# Kaç işlem seçileceği
SAMPLE_SIZE = 100
RANDOM_STATE = 42

# Proje kökünü bul
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Dosya yolları
test_path = (
    project_root
    / "data"
    / "processed"
    / "catboost"
    / "test.parquet"
)

demo_dir = project_root / "data" / "demo"
demo_dir.mkdir(parents=True, exist_ok=True)

output_path = demo_dir / "ecoshield_random_test_demo_w100.csv"

# Test verisini yükle
print(f"Test verisi yükleniyor: {test_path}")

test_df = pd.read_parquet(test_path)

print(f"Test veri boyutu: {test_df.shape}")
print(f"Rastgele seçilecek işlem sayısı: {SAMPLE_SIZE}")

if len(test_df) < SAMPLE_SIZE:
    raise ValueError(
        f"Test verisinde {len(test_df):,} satır var; "
        f"{SAMPLE_SIZE:,} satır seçilemez."
    )

# Test verisinin tamamından rastgele seçim
demo_df = test_df.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    replace=False,
).copy()

# Hedef kolonu yanlışlıkla bulunuyorsa kaldır
if "isFraud" in demo_df.columns:
    demo_df = demo_df.drop(columns=["isFraud"])

# Eski DataFrame indexini kaldır
demo_df = demo_df.reset_index(drop=True)

# CSV olarak kaydet
demo_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nDemo CSV başarıyla oluşturuldu.")
print(f"Dosya: {output_path}")
print(f"Satır sayısı: {len(demo_df):,}")
print(f"Kolon sayısı: {demo_df.shape[1]:,}")

if "TransactionID" in demo_df.columns:
    print("\nSeçilen TransactionID değerleri:")
    print(demo_df["TransactionID"].to_string(index=False))

display(demo_df.head())

Test verisi yükleniyor: c:\Users\pc\Desktop\YZTA-Bootcamp-2026\data\processed\catboost\test.parquet
Test veri boyutu: (88581, 423)
Rastgele seçilecek işlem sayısı: 30

Demo CSV başarıyla oluşturuldu.
Dosya: c:\Users\pc\Desktop\YZTA-Bootcamp-2026\data\demo\ecoshield_random_test_demo.csv
Satır sayısı: 30
Kolon sayısı: 423


,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,3362580,82.95,W,18343,374.0,150.0,visa,226.0,debit,315.0,...,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__
1,5941342,25.95,W,16560,476.0,150.0,visa,166.0,debit,225.0,...,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__
2,14923668,80.00,W,18129,321.0,150.0,visa,226.0,debit,126.0,...,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__
3,1792259,200.00,R,10616,583.0,150.0,visa,226.0,credit,299.0,...,mobile safari 11.0,32.0,2208x1242,match_status:1,T,F,F,F,mobile,iOS Device
4,15535191,511.95,W,15066,170.0,150.0,mastercard,102.0,credit,315.0,...,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__
